In [1]:
import numpy as np  
import pandas as pd

In [105]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv(r"C:\Users\gsksd\Downloads\covid_toy.csv")

In [3]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
35,82,Female,102.0,Strong,Bangalore,No
32,34,Female,101.0,Strong,Delhi,Yes
97,20,Female,101.0,Mild,Bangalore,No
3,31,Female,98.0,Mild,Kolkata,No
36,38,Female,101.0,Mild,Bangalore,No


In [4]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [6]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [7]:
df['has_covid'].value_counts()

has_covid
No     55
Yes    45
Name: count, dtype: int64

In [8]:
df['gender'].value_counts()

gender
Female    59
Male      41
Name: count, dtype: int64

In [27]:
X = df.iloc[:,0:5]
y = df.iloc[:,-1]

In [28]:
X

,age,gender,fever,cough,city
0,60,Male,103.0,Mild,Kolkata
1,27,Male,100.0,Mild,Delhi
2,42,Male,101.0,Mild,Delhi
3,31,Female,98.0,Mild,Kolkata
4,65,Female,101.0,Mild,Mumbai
...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore
96,51,Female,101.0,Strong,Kolkata
97,20,Female,101.0,Mild,Bangalore
98,5,Female,98.0,Strong,Mumbai


In [29]:
y

0      No
1     Yes
2      No
3      No
4      No
     ... 
95     No
96    Yes
97     No
98     No
99    Yes
Name: has_covid, Length: 100, dtype: object

## Splitting Train Test data 

In [42]:
X_test , X_train , y_test , y_train = train_test_split(X,y,train_size=0.2)

In [43]:
X_train.shape

(80, 5)

In [46]:
y_train.shape

(80,)

## without using Column Transformer

In [47]:
si = SimpleImputer()

In [48]:
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.fit_transform(X_test[['fever']])

In [52]:
X_train_fever.shape


(80, 1)

In [53]:
oe = OrdinalEncoder(categories=[['Mild' , 'Strong']])

In [75]:
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.fit_transform(X_test[['cough']])

In [77]:
X_train_cough.shape  

(80, 1)

In [78]:
ohe = OneHotEncoder(drop='first' , sparse_output=False)

In [79]:
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])
X_train_gender_city.shape

(80, 4)

In [80]:
X_train_gender_city

array([[0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.],
       [1., 0., 1., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 0.],
       [1., 0., 0., 1.],
       [0., 1., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 1., 0.],
       [0., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 0.],
       [1., 0., 1., 0.],
       [0., 0., 0., 0.],
       [0., 0., 1., 0.],
       [1., 0., 1., 0.],
       [1., 0., 1., 0.],
       [1., 1., 0., 0.],
       [0., 1., 0., 0.],
       [1., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [1., 0., 0., 1.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 0.],
       [1., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 1., 0.],
       [1., 1., 0., 0.],


In [81]:
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values
X_train_age.shape

(80, 1)

In [84]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)

X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

## Using Column Transformer

In [91]:
from sklearn.compose import ColumnTransformer

In [108]:
transformer = ColumnTransformer(transformers=[
    ('tn1' , SimpleImputer(),['fever']), 
    ('tn2' , OrdinalEncoder(categories=[['Mild' , 'Strong']]),['cough']), 
    ('tn3' , OneHotEncoder(drop='first' , sparse_output=False),['gender','city'])
] , remainder='passthrough')

In [109]:
X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)

In [110]:
transformer.fit_transform(X_train).shape

(80, 7)

In [118]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [119]:
y_train

array([0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0,
       0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0])

In [120]:
X_train_df = pd.DataFrame(X_train_transformed)
X_test_df = pd.DataFrame(X_test_transformed)

In [123]:
train_df = pd.concat(
    [X_train_df, pd.DataFrame(y_train, columns=['has_covid'])],
    axis=1
)

test_df = pd.concat(
    [X_test_df, pd.DataFrame(y_test, columns=['has_covid'])],
    axis=1
)

In [124]:
combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

combined_df.to_csv("combined.csv", index=False)

In [125]:
combined_df.sample(5)

,0,1,2,3,4,5,6,has_covid
80,99.0,0.0,0.0,0.0,0.0,1.0,60.0,1
33,102.0,1.0,0.0,0.0,0.0,0.0,24.0,1
22,101.0,0.0,0.0,0.0,1.0,0.0,83.0,0
64,101.0,0.0,0.0,1.0,0.0,0.0,49.0,1
41,98.0,0.0,1.0,0.0,1.0,0.0,24.0,1


In [126]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

# Split the data
X = df.drop(columns=['has_covid'])
y = df['has_covid']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create the ColumnTransformer
transformer = ColumnTransformer(
    transformers=[
        ('tn1', SimpleImputer(strategy='most_frequent'), ['fever']),
        ('tn2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('tn3', OneHotEncoder(drop='first', sparse_output=False), ['gender', 'city'])
    ],
    remainder='passthrough'
)

# Fit on train and transform
X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)

# Get transformed column names
column_names = transformer.get_feature_names_out()

# Convert to DataFrames
X_train_df = pd.DataFrame(
    X_train_transformed,
    columns=column_names
)

X_test_df = pd.DataFrame(
    X_test_transformed,
    columns=column_names
)

# Convert y to DataFrame
y_train = y_train.reset_index(drop=True).to_frame(name='has_covid')
y_test = y_test.reset_index(drop=True).to_frame(name='has_covid')

# Concatenate X and y
train_df = pd.concat(
    [X_train_df.reset_index(drop=True), y_train],
    axis=1
)

test_df = pd.concat(
    [X_test_df.reset_index(drop=True), y_test],
    axis=1
)

# Save CSV files
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

# Preview
print(train_df.head())
print(test_df.head())

   tn1__fever  tn2__cough  tn3__gender_Male  tn3__city_Delhi  \
0       101.0         0.0               0.0              0.0   
1       100.0         0.0               0.0              0.0   
2       100.0         0.0               0.0              0.0   
3       100.0         0.0               1.0              1.0   
4       103.0         0.0               0.0              1.0   

   tn3__city_Kolkata  tn3__city_Mumbai  remainder__age has_covid  
0                0.0               1.0            81.0       Yes  
1                1.0               0.0             5.0        No  
2                1.0               0.0            19.0       Yes  
3                0.0               0.0            27.0       Yes  
4                0.0               0.0            73.0        No  
   tn1__fever  tn2__cough  tn3__gender_Male  tn3__city_Delhi  \
0       104.0         0.0               0.0              0.0   
1        98.0         0.0               1.0              1.0   
2       101.0        